<a href="https://colab.research.google.com/github/braysonimbambi455/CSC4110-G03/blob/main/notebooks/G03_U2.1_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 2.1 — Data inspection and cleaning**  
**Owner:** Brayson Imbambi (COM/0060/23)  
**Module:** 3

In [ ]:
# Mount Google Drive so we can read/write our project folder
from google.colab import drive
drive.mount('/content/drive')

## (a) Inspect the data

In [ ]:
# Load the raw dataset from Drive
import pandas as pd, numpy as np
df = pd.read_csv("/content/drive/MyDrive/CSC4110_G03/data/yield_df.csv")

# Keep an untouched copy of the raw data for comparison
df_raw = df.copy()

In [ ]:
# Show column names, dtypes, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28242 entries, 0 to 28241
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     28242 non-null  int64  
 1   Area                           28242 non-null  object 
 2   Item                           28242 non-null  object 
 3   Year                           28242 non-null  int64  
 4   hg/ha_yield                    28242 non-null  int64  
 5   average_rain_fall_mm_per_year  28242 non-null  float64
 6   pesticides_tonnes              28242 non-null  float64
 7   avg_temp                       28242 non-null  float64
dtypes: float64(3), int64(3), object(2)
memory usage: 1.7+ MB


In [ ]:
# Count missing values in each column
print(df.isna().sum())

# Count duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Unnamed: 0                       0
Area                             0
Item                             0
Year                             0
hg/ha_yield                      0
average_rain_fall_mm_per_year    0
pesticides_tonnes                0
avg_temp                         0
dtype: int64
Duplicate rows: 0


In [ ]:
# Show summary statistics for numeric columns
df.describe()

,Unnamed: 0,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
count,28242.000000,28242.000000,28242.000000,28242.00000,28242.000000,28242.000000
mean,14120.500000,2001.544296,77053.332094,1149.05598,37076.909344,20.542627
std,8152.907488,7.051905,84956.612897,709.81215,59958.784665,6.312051
min,0.000000,1990.000000,50.000000,51.00000,0.040000,1.300000
25%,7060.250000,1995.000000,19919.250000,593.00000,1702.000000,16.702500
50%,14120.500000,2001.000000,38295.000000,1083.00000,17529.440000,21.510000
75%,21180.750000,2008.000000,104676.750000,1668.00000,48687.880000,26.000000
max,28241.000000,2013.000000,501412.000000,3240.00000,367778.000000,30.650000


In [ ]:
# Show how many rows belong to each crop type
print(df["Item"].value_counts())

Item
Potatoes                4276
Maize                   4121
Wheat                   3857
Rice, paddy             3388
Soybeans                3223
Sorghum                 3039
Sweet potatoes          2890
Cassava                 2045
Yams                     847
Plantains and others     556
Name: count, dtype: int64


## (b) Clean the data

In [ ]:
# Drop the useless row-number column and remove exact duplicate rows
before = len(df)
df = df.drop(columns=["Unnamed: 0"])
df = df.drop_duplicates()
print("Rows removed as duplicates:", before - len(df))

Rows removed as duplicates: 2310


In [ ]:
# Rename long column names to shorter, easier-to-type versions
df = df.rename(columns={
    "hg/ha_yield":"yield_hg_ha",
    "average_rain_fall_mm_per_year":"rain_mm",
    "pesticides_tonnes":"pesticides_t"
})
print(df.columns.tolist())

['Area', 'Item', 'Year', 'yield_hg_ha', 'rain_mm', 'pesticides_t', 'avg_temp']


In [ ]:
# Check for impossible values: negative rainfall, negative pesticides, extreme temperature
print("Negative rainfall rows:", (df.rain_mm < 0).sum())
print("Negative pesticides rows:", (df.pesticides_t < 0).sum())
print("Temperature outside -10 to 45 C:", ((df.avg_temp < -10) | (df.avg_temp > 45)).sum())

Negative rainfall rows: 0
Negative pesticides rows: 0
Temperature outside -10 to 45 C: 0


In [ ]:
# Apply the IQR rule to flag/cap pesticide outliers
outlier_col = "pesticides_t"
Q1 = df[outlier_col].quantile(0.25)
Q3 = df[outlier_col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Rows below lower:", (df[outlier_col] < lower).sum())
print("Rows above upper:", (df[outlier_col] > upper).sum())
# Cap values to the bounds (very large values may be real, e.g. large farming countries,
# so we cap rather than delete)
df[outlier_col] = df[outlier_col].clip(lower=lower, upper=upper)

Lower bound: -64437.5
Upper bound: 111654.5
Rows below lower: 0
Rows above upper: 1262


In [ ]:
# Fix wrong data types: rain_mm may be stored as text like "123 mm"
df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
print(df.dtypes)

Area             object
Item             object
Year              int64
yield_hg_ha       int64
rain_mm         float64
pesticides_t    float64
avg_temp        float64
dtype: object


In [ ]:
# Save the final cleaned dataframe as df_stage1
df_stage1 = df.copy()
print("Final shape:", df_stage1.shape)

Final shape: (25932, 7)


(c) Exam skill — median imputation by hand

In [ ]:
# Take first 8 rain_mm values, set the 3rd to missing on a COPY
sample = df_stage1["rain_mm"].head(8).copy()
sample.iloc[2] = None
print("With missing:")
print(sample.tolist())

With missing:
[1485.0, 1485.0, nan, 1485.0, 1485.0, 1485.0, 1485.0, 1485.0]


In [ ]:
# Show the sorted non-missing values so we can see the middle
sorted_vals = sorted(sample.dropna().tolist())
print("Sorted non-missing values:", sorted_vals)
print("Number of non-missing values:", len(sorted_vals))
# With 7 values, the median is the 4th one
median_manual = sorted_vals[3]
print("Median by hand:", median_manual)

Sorted non-missing values: [1485.0, 1485.0, 1485.0, 1485.0, 1485.0, 1485.0, 1485.0]
Number of non-missing values: 7
Median by hand: 1485.0


In [ ]:
# Fill the missing value using pandas fillna with the median
filled = sample.fillna(median_manual)
print("After fillna:", filled.tolist())

After fillna: [1485.0, 1485.0, 1485.0, 1485.0, 1485.0, 1485.0, 1485.0, 1485.0]


The missing value in position 3 is replaced with the median of the other 7 values.
The median is the middle number when the values are sorted. Using the median (not the
mean) is safer because it is not pulled by very large or very small outliers.
`sample.fillna(median_manual)` replaces every NaN in `sample` with the value we computed.

# Written answer — cleaning decisions

I made these cleaning decisions:

1. Dropped the `Unnamed: 0` column — it was just a row number with no predictive value.
2. Removed duplicate rows — there were ___ duplicates (count from output above), which
   would have inflated the training set and biased the model.
3. Renamed `hg/ha_yield → yield_hg_ha`, `average_rain_fall_mm_per_year → rain_mm`,
   `pesticides_tonnes → pesticides_t` so downstream code is cleaner.
4. Capped `pesticides_t` outliers using the IQR rule (lower = ___, upper = ___). I capped
   instead of removing because very large pesticide values may be genuine for large
   farming countries like China, Brazil and the USA; removing them would lose real data.
5. Converted `rain_mm` to numeric using `pd.to_numeric(errors="coerce")` — some values
   were stored as text like "123 mm".
6. I did NOT drop rows with any remaining missing values because ___ (state whether any
   remain after the coerce).

Total rows before: ___, after: ___. Rows removed: ___.

In [ ]:
# Save the cleaned dataset to Drive for use in later tasks
df_stage1.to_csv("/content/drive/MyDrive/CSC4110_G03/data/stage1_crops.csv", index=False)
print("Saved to data/stage1_crops.csv")

Saved to data/stage1_crops.csv


In [ ]:
# Self-check: confirm all required graded-evidence variables exist
for name in ["df_raw", "df_stage1", "outlier_col", "lower", "upper"]:
    print(name, "->", "OK" if name in dir() else "MISSING")

df_raw -> OK
df_stage1 -> OK
outlier_col -> OK
lower -> OK
upper -> OK
